### Import Libraries

In [3]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
from datasets import load_dataset


C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Import Dataset and Configure

In [2]:
dataset = load_dataset("ag_news")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [5]:
dataset['train'][1]

{'text': 'Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\\which has a reputation for making well-timed and occasionally\\controversial plays in the defense industry, has quietly placed\\its bets on another part of the market.',
 'label': 2}

In [7]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [6]:
ag_news_labels = {0: "world", 1: "sport",2: "business", 3: "sci/tec"}

In [17]:
def tokenized_text (mydataset):
    
    return tokenizer(mydataset['text'], padding="max_length", max_length=128, truncation=True)

In [18]:
dataset_tokenized = dataset.map(tokenized_text, batched=True)

Map: 100%|██████████| 7600/7600 [00:00<00:00, 7887.70 examples/s]


In [19]:
dataset_tokenized

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7600
    })
})

In [20]:
dataset_tokenized = dataset_tokenized.rename_column("label", "labels")

In [21]:
dataset_tokenized.set_format(type='torch', columns=['labels','input_ids', 'attention_mask'])

In [22]:
dataset_tokenized['train'][2]

{'labels': tensor(2),
 'input_ids': tensor([  101,  3514,  1998,  4610,  6112, 15768,  1005, 17680,  1006, 26665,
          1007, 26665,  1011, 23990, 13587,  7597,  4606, 15508,  1032,  2055,
          1996,  4610,  1998,  1996, 17680,  2005, 16565,  2024,  3517,  2000,
          1032,  6865,  2058,  1996,  4518,  3006,  2279,  2733,  2076,  1996,
          5995,  1997,  1996,  1032,  2621,  2079,  6392,  6824,  2015,  1012,
           102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,  

In [23]:
dataset_train = dataset_tokenized['train']
dataset_test = dataset_tokenized['test']

In [24]:
dataset_train

Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 120000
})

In [25]:
train_test_split = dataset_train.train_test_split(test_size=0.1, seed = 42)
dataset_train = train_test_split['train']
dataset_val = train_test_split['test']

In [26]:
dataset_train

Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 108000
})

### Token Embedding

In [28]:
class TokenEmbeddings(nn.Module):
    
    def __init__(self, vocab_size, d_model, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.embeddings = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model
    
    def forward(self, input_ids):
        # input_ids -> [batch_size, seq_len]
        
        embed_out = self.embeddings(input_ids) * math.sqrt(self.d_model)
        return self.dropout(embed_out)

### Positional Embeddings

In [29]:
class PositionalEmbeddings (nn.Module):
    
    def __init__(self, max_seq_len, d_model):
        super().__init__()
        
        positions = torch.arange(0, max_seq_len).unsqueeze(1)
        
        pe = torch.zeros(max_seq_len, d_model)
        
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000)/d_model)
        )
        
        pe[:, 0::2] = torch.sin(positions * div_term)
        pe[:, 1::2] = torch.cos(positions * div_term)
        
        
        pe = pe.unsqueeze(0)
        
        self.register_buffer("pe", pe)
        
    def forward(self, token_embeddings):
        
        seq_len = token_embeddings.size(1)
        return token_embeddings + self.pe[:,0:seq_len, :]
        

### Rotary Positional Embeddings

**1. Core RoPE: Precompute Rotation Frequencies**

 Precompute complex rotation frequencies for RoPE.
 
    For each dimension pair (i, i+1), we compute a frequency:
        freq_i = 1 / (theta ^ (2i / dim))
 
    Then for each position m, we get an angle:
        angle = m * freq_i
 
    Returned as complex numbers: e^(i * angle) = cos(angle) + i*sin(angle)
 
    Args:
        dim:     head dimension (must be even)
        seq_len: maximum sequence length to precompute
        theta:   base for frequency scaling (default 10000, LLaMA uses this)
 
    Returns:
        freqs_cis: [seq_len, dim//2] complex tensor

In [ ]:
def precompute_freq_angle(seq_len:int, d_model:int, theta:float = 10000.0) -> torch.Tensor:
    
    # Shape: [dim//2]
    # freqs[i] = 1 / (10000 ^ (2i / dim))
    
    freqs = 1/ (theta ** (torch.arange(0, d_model, 2))/d_model)
    
    positions = torch.arange(0, seq_len) #shape: [seq_len]
    
    # angle = positons * freqs
    angles = torch.outer(positions, freqs) # [seq_len, dim//2]
    
    # Convert to complex: cos(angle) + i*sin(angle)``
    # Shape: [seq_len, dim//2]
    freq_cis = torch.polar(torch.ones_like(angles), angles)
    #This just creates a tensor of all 1s with the exact same shape as
    #Why all 1s specifically?
        #Because we want pure rotation — no stretching, no shrinking.
    
    return freq_cis    

**2. Apply Rotation to Q or K Tensors**


    Apply rotary embeddings to a query or key tensor.
 
    The trick: treat each pair of (x_{2i}, x_{2i+1}) as a complex number,
    then multiply by the rotation e^(i * m * freq_i).
 
    This effectively rotates the vector in 2D subspaces, encoding position
    *relative* to the current token.
 
    Args:
        x:          [batch, seq_len, n_heads, head_dim]
        freqs_cis:  [seq_len, head_dim//2]  (complex)
 
    Returns:
        x_rotated:  [batch, seq_len, n_heads, head_dim]
    

In [ ]:
def apply_rotary_embedding(x:torch.Tensor, freq_cis:torch.Tensor):
    
    x_complex = torch.view_as_complex(x.float().reshape(x.shape[:-1],-1,2))
    
    freq_cis = freq_cis.unsqueeze(0).unsqueeze(2)
    
    x_rotated = x_complex * freq_cis
    
    

In [6]:
x = torch.tensor([2, 16, 8, 64])
x.shape

torch.Size([4])

In [8]:
x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1],-1,2))
x_complex.shape, x_complex

(torch.Size([2]), tensor([2.+16.j, 8.+64.j]))

In [ ]:
def precompute_freq_angle (d_model, seq_len, theata = 10000):
    
    freq =  1 / (theata ** (torch.arange(0,d_model, 2)/d_model))
    
    
    positions = torch.arange (0, seq_len)
    
    angles = torch.outer(positions, freq)
    
    freq_complex = torch.polar(torch.ones_like(angles), angles)
    # [seq_len, dim//2]
    
    return freq_complex

def apply_rotary_embed ( x:torch.Tensor, freq_complex:torch.Tensor):
    
    # shape: [batch_Size, seq_len, nheads, ndims]
    x_complex = torch.view_as_complex(x.float().reshape(x.shape[:-1],-1,2))
    
    
    freq_complex =   freq_complex.unsqueeze(0).unsqueeze(2)
    
    x_rotated = x_complex * freq_complex 
    
    x_out = torch.view_as_real(x_rotated).flatten(3)
    
    return x_out.type_as(x)